# Absenteeism Prediction — Model HR Kedua

Use case **#3** di `USE_CASES.md`: memprediksi karyawan yang berisiko
**absen tinggi** (masalah kehadiran) agar bisa diintervensi lebih awal.

> **Data:** `data/hr_absenteeism.csv` adalah dataset **sintetis** (dibuat oleh
> `scripts/generate_absenteeism_data.py`). Skemanya meniru data absensi nyata
> tetapi tidak merepresentasikan orang sungguhan.

## Ringkasan alur

1. Load & audit data
2. EDA singkat
3. Feature engineering + pipeline preprocessing
4. Perbandingan beberapa model (cross-validation)
5. Tuning model teratas
6. Pemilihan threshold dari OOF
7. Evaluasi holdout
8. Interpretasi (permutation importance)
9. Simpan artifact ke `models/absenteeism/v1/`

## 0. Import & Konfigurasi

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, RepeatedStratifiedKFold, cross_val_score, GridSearchCV,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, HistGradientBoostingClassifier,
    GradientBoostingClassifier, ExtraTreesClassifier,
)
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    precision_score, recall_score, f1_score, classification_report,
    precision_recall_curve, confusion_matrix,
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

CANDIDATE_PATHS = ["data/hr_absenteeism.csv", "../data/hr_absenteeism.csv", "hr_absenteeism.csv"]
DATA_PATH = next((Path(p) for p in CANDIDATE_PATHS if Path(p).exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("hr_absenteeism.csv tidak ditemukan.")

CWD = Path.cwd()
PROJECT_ROOT = CWD if (CWD / "data").exists() else CWD.parent
ARTIFACT_DIR = PROJECT_ROOT / "models" / "absenteeism" / "v1"

TARGET = "target_absen_tinggi"

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 160)
print("Data     :", DATA_PATH)
print("Artifact :", ARTIFACT_DIR)

Data     : data/hr_absenteeism.csv
Artifact : /workspace/project/hr-intelligence-system/models/absenteeism/v1


## 1. Load & Audit Data

Kolom yang **tidak** dipakai sebagai fitur (kebocoran / bukan prediktor):
`employee_id`, `hari_absen_setahun` (ini sumber target — memakainya = leakage),
`alasan_dominan` (dicatat setelah absen terjadi).

In [2]:
df = pd.read_csv(DATA_PATH, parse_dates=["tanggal_masuk"])

LEAKAGE_COLS = ["employee_id", "hari_absen_setahun", "alasan_dominan"]
# tanggal_masuk redundan dengan masa_kerja_bulan (dan ber-kardinalitas tinggi).
DROP_COLS = ["tanggal_masuk"]
FEATURES = [c for c in df.columns if c not in LEAKAGE_COLS + DROP_COLS + [TARGET]]

NUM_COLS = [c for c in FEATURES if pd.api.types.is_numeric_dtype(df[c])]
CAT_COLS = [c for c in FEATURES if c not in NUM_COLS]

print("Baris   :", len(df))
print("Fitur   :", len(FEATURES), "->", len(NUM_COLS), "numerik,", len(CAT_COLS), "kategorikal")
print("Numerik :", NUM_COLS)
print("Kategori:", CAT_COLS)
print("\nMissing :", int(df[FEATURES].isna().sum().sum()))
print("Rasio target:", round(float(df[TARGET].mean()), 4))
df[FEATURES + [TARGET]].head(3)

Baris   : 1200
Fitur   : 10 -> 7 numerik, 3 kategorikal
Numerik : ['usia', 'masa_kerja_bulan', 'jarak_km', 'beban_kerja', 'lembur_jam_bulanan', 'kepuasan_kerja', 'riwayat_absen_90hari']
Kategori: ['departemen', 'status_kerja', 'pendidikan']

Missing : 0
Rasio target: 0.1683


,usia,masa_kerja_bulan,departemen,status_kerja,pendidikan,jarak_km,beban_kerja,lembur_jam_bulanan,kepuasan_kerja,riwayat_absen_90hari,target_absen_tinggi
0,42,46,Sales,Tetap,S1,16.2,59.6,33.3,4.55,2,0
1,28,32,Finance,Kontrak,S1,1.8,83.3,12.5,4.87,5,0
2,19,54,IT,Freelance,SMA,4.5,48.7,4.7,3.90,2,0


## 2. EDA Singkat

In [3]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
df[TARGET].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="#4C78A8")
axes[0].set_title("Distribusi target"); axes[0].set_xticklabels(["Tidak (0)", "Ya (1)"], rotation=0)

df.groupby("departemen")[TARGET].mean().sort_values().plot(kind="barh", ax=axes[1], color="#F58518")
axes[1].set_title("Rasio absen tinggi per departemen")

sns.boxplot(data=df, x=TARGET, y="riwayat_absen_90hari", ax=axes[2])
axes[2].set_title("Riwayat absen 90 hari vs target")
plt.tight_layout(); plt.show()

print("Rata-rata hari absen per target:")
print(df.groupby(TARGET)["hari_absen_setahun"].agg(["mean", "count"]).round(2))

Rata-rata hari absen per target:
                      mean  count
target_absen_tinggi              
0                     4.67    998
1                    13.39    202


## 3. Split & Pipeline Preprocessing

Split **stratified** dan pipeline di-*fit* hanya pada data train untuk mencegah
kebocoran. `StandardScaler` dipakai agar Logistic Regression stabil.

In [4]:
X = df[FEATURES]
y = df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), NUM_COLS),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), CAT_COLS),
])

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Rasio target train:", round(float(y_train.mean()), 4), "| test:", round(float(y_test.mean()), 4))

Train: (900, 10) | Test: (300, 10)
Rasio target train: 0.1689 | test: 0.1667


## 4. Perbandingan Model (Cross-Validation)

Model dibandingkan dengan `RepeatedStratifiedKFold` pada data train.
ROC-AUC untuk peringkat, PR-AUC karena kelas tidak seimbang.

In [5]:
candidates = {
    "Dummy (prior)": DummyClassifier(strategy="prior"),
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=400, random_state=SEED, n_jobs=-1),
    "Extra Trees": ExtraTreesClassifier(n_estimators=400, random_state=SEED, n_jobs=-1),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(random_state=SEED),
    "Gradient Boosting": GradientBoostingClassifier(random_state=SEED),
}

try:
    from xgboost import XGBClassifier
    candidates["XGBoost"] = XGBClassifier(
        n_estimators=400, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
        random_state=SEED, n_jobs=4)
except Exception as exc:
    print("XGBoost tidak tersedia:", exc)

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
rows = []
for name, clf in candidates.items():
    pipe = Pipeline([("prep", preprocess), ("clf", clf)])
    auc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=1)
    ap = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="average_precision", n_jobs=1)
    rows.append({"model": name, "roc_auc_cv": auc.mean(), "roc_auc_std": auc.std(),
                 "pr_auc_cv": ap.mean(), "pr_auc_std": ap.std()})

hasil_cv = pd.DataFrame(rows).sort_values("roc_auc_cv", ascending=False).reset_index(drop=True)
print(hasil_cv.round(4).to_string(index=False))

                 model  roc_auc_cv  roc_auc_std  pr_auc_cv  pr_auc_std
   Logistic Regression      0.7964       0.0408     0.4544      0.0755
     Gradient Boosting      0.7462       0.0378     0.3885      0.0679
         Random Forest      0.7431       0.0382     0.3941      0.0710
               XGBoost      0.7320       0.0385     0.3718      0.0716
Hist Gradient Boosting      0.7214       0.0304     0.3556      0.0589
           Extra Trees      0.6889       0.0454     0.3301      0.0455
         Dummy (prior)      0.5000       0.0000     0.1689      0.0027


## 5. Tuning Model Teratas

Model non-dummy terbaik di-tuning dengan `GridSearchCV` (ROC-AUC).
Grid sengaja kecil agar notebook tetap cepat dijalankan.

In [6]:
top_name = hasil_cv[hasil_cv["model"] != "Dummy (prior)"].iloc[0]["model"]
print("Model teratas:", top_name)

GRIDS = {
    "Logistic Regression": {
        "clf__C": [0.01, 0.1, 1.0, 10.0],
        "clf__penalty": ["l2"],
    },
    "Random Forest": {
        "clf__n_estimators": [300, 600],
        "clf__max_depth": [None, 4, 8],
        "clf__min_samples_leaf": [1, 5],
    },
    "Extra Trees": {
        "clf__n_estimators": [300, 600],
        "clf__max_depth": [None, 4, 8],
    },
    "Hist Gradient Boosting": {
        "clf__learning_rate": [0.03, 0.1],
        "clf__max_depth": [None, 3],
        "clf__max_iter": [200, 400],
    },
    "Gradient Boosting": {
        "clf__learning_rate": [0.03, 0.1],
        "clf__n_estimators": [200, 400],
        "clf__max_depth": [2, 3],
    },
    "XGBoost": {
        "clf__n_estimators": [300, 600],
        "clf__max_depth": [3, 5],
        "clf__learning_rate": [0.03, 0.1],
    },
}

base_pipe = Pipeline([("prep", preprocess), ("clf", candidates[top_name])])
grid = GRIDS.get(top_name, {})
if grid:
    search = GridSearchCV(base_pipe, grid, scoring="roc_auc",
                          cv=StratifiedKFold(5, shuffle=True, random_state=SEED), n_jobs=1)
    search.fit(X_train, y_train)
    best_pipe = search.best_estimator_
    print("Best params:", search.best_params_)
    print("Best CV ROC-AUC:", round(search.best_score_, 4))
else:
    best_pipe = base_pipe.fit(X_train, y_train)
    print("Tidak ada grid, memakai default.")

Model teratas: Logistic Regression


Best params: {'clf__C': 0.01, 'clf__penalty': 'l2'}
Best CV ROC-AUC: 0.8042


## 6. Pemilihan Threshold dari Out-of-Fold (OOF)

Threshold dipilih untuk memaksimalkan F1 pada prediksi OOF — bukan pada test set,
supaya tidak ada kebocoran.

In [7]:
from sklearn.model_selection import cross_val_predict

oof = cross_val_predict(best_pipe, X_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                        method="predict_proba", n_jobs=1)[:, 1]

prec, rec, thr = precision_recall_curve(y_train, oof)
f1 = 2 * prec * rec / np.clip(prec + rec, 1e-12, None)
best_idx = int(np.nanargmax(f1[:-1]))
best_threshold = float(thr[best_idx])
print(f"Threshold terpilih: {best_threshold:.3f}  (F1 OOF = {f1[best_idx]:.4f})")

Threshold terpilih: 0.193  (F1 OOF = 0.4964)


## 7. Evaluasi Holdout

Metrik pada test set yang **belum pernah dilihat** model.

In [8]:
proba_test = best_pipe.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= best_threshold).astype(int)

metrics = {
    "roc_auc": roc_auc_score(y_test, proba_test),
    "pr_auc": average_precision_score(y_test, proba_test),
    "brier": brier_score_loss(y_test, proba_test),
    "precision": precision_score(y_test, pred_test, zero_division=0),
    "recall": recall_score(y_test, pred_test, zero_division=0),
    "f1": f1_score(y_test, pred_test, zero_division=0),
    "threshold": best_threshold,
}
for k, v in metrics.items():
    print(f"{k:>10}: {v:.4f}")

print("\n", classification_report(y_test, pred_test, target_names=["Tidak", "Ya"], zero_division=0))

tn, fp, fn, tp = confusion_matrix(y_test, pred_test, labels=[0, 1]).ravel()
print(f"TN={tn} FP={fp} FN={fn} TP={tp}")

   roc_auc: 0.8287
    pr_auc: 0.4974
     brier: 0.1148
 precision: 0.3548
    recall: 0.6600
        f1: 0.4615
 threshold: 0.1929

               precision    recall  f1-score   support

       Tidak       0.92      0.76      0.83       250
          Ya       0.35      0.66      0.46        50

    accuracy                           0.74       300
   macro avg       0.64      0.71      0.65       300
weighted avg       0.82      0.74      0.77       300

TN=190 FP=60 FN=17 TP=33


## 8. Interpretasi — Permutation Importance

Diukur pada test set: seberapa besar ROC-AUC turun bila sebuah fitur diacak.

In [9]:
perm = permutation_importance(best_pipe, X_test, y_test, scoring="roc_auc",
                              n_repeats=10, random_state=SEED, n_jobs=1)
imp = pd.DataFrame({
    "fitur": X_test.columns,
    "turunnya_roc_auc": perm.importances_mean,
    "std": perm.importances_std,
}).sort_values("turunnya_roc_auc", ascending=False).reset_index(drop=True)

print(imp.round(4).to_string(index=False))

plt.figure(figsize=(8, 5))
sns.barplot(data=imp.head(10), x="turunnya_roc_auc", y="fitur", color="#4C78A8")
plt.title("Permutation importance (ROC-AUC, test set)")
plt.tight_layout(); plt.show()

               fitur  turunnya_roc_auc    std
            jarak_km            0.0962 0.0179
riwayat_absen_90hari            0.0889 0.0242
  lembur_jam_bulanan            0.0695 0.0122
      kepuasan_kerja            0.0435 0.0167
         beban_kerja            0.0223 0.0123
    masa_kerja_bulan            0.0037 0.0028
          pendidikan            0.0018 0.0023
                usia            0.0008 0.0010
        status_kerja            0.0005 0.0009
          departemen            0.0001 0.0016


## 9. Simpan Artifact

Artifact disimpan ke `models/absenteeism/v1/` mengikuti pola versioning
`models/README.md`.

In [10]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(best_pipe, ARTIFACT_DIR / "absenteeism_model.joblib")

metadata = {
    "model": top_name,
    "versi_model": "v1",
    "target": TARGET,
    "threshold": best_threshold,
    "fitur_numerik": NUM_COLS,
    "fitur_kategorikal": CAT_COLS,
    "training_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "dilatih_pada": pd.Timestamp.utcnow().isoformat(),
    "metrik_holdout_test": {k: round(float(v), 4) for k, v in metrics.items()},
    "dataset": str(DATA_PATH),
    "dataset_sintetis": True,
}
(ARTIFACT_DIR / "model_metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")

print("Tersimpan di:", ARTIFACT_DIR)
print(json.dumps(metadata["metrik_holdout_test"], indent=2))

Tersimpan di: /workspace/project/hr-intelligence-system/models/absenteeism/v1
{
  "roc_auc": 0.8287,
  "pr_auc": 0.4974,
  "brier": 0.1148,
  "precision": 0.3548,
  "recall": 0.66,
  "f1": 0.4615,
  "threshold": 0.1929
}
